# Smartwatch Activity Analysis – Identify Activity Behavior Groups

Developed using Unsupervised Machine Learning (K-Means Clustering & Principal Component Analysis - PCA).

### Project Objective:
Develop a machine learning application that analyzes wearable/smartwatch sensor data and identifies different activity behavior groups using unsupervised machine learning.

The notebook performs:
1. Data Loading & Preprocessing
2. Exploratory Data Analysis (EDA)
3. Standard Scaling & PCA Dimensionality Reduction
4. Elbow Method for Optimal K Selection
5. K-Means Clustering Training
6. Activity Behavior Group Identification & Profiling
7. Model Artifact Serialization

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import joblib

# Add root directory to sys.path to import src modules
sys.path.append("..")
from src.preprocessing import load_dataset, preprocess_data, scale_features
from src.clustering import compute_elbow, compute_pca, train_kmeans, save_ml_artifacts
from src.analysis import profile_clusters, select_important_features


## 1. Data Preprocessing & Column Detection

In [ ]:
data_path = "../data/dataset.csv"
df = load_dataset(data_path)
print("Loaded Dataset Shape:", df.shape)

df_clean, X, summary = preprocess_data(df)
print("Cleaned Shape:", df_clean.shape)
print("Target Column Detected:", summary["target_col"])
print("ID Columns Detected:", summary["id_cols"])
print("Sensor Numerical Columns Count:", len(summary["sensor_cols"]))


## 2. Feature Scaling & PCA Dimensionality Reduction

In [ ]:
X_scaled, scaler = scale_features(X)
print("Scaled Feature Matrix Shape:", X_scaled.shape)

pca_df, pca_model, var_ratio, cum_var = compute_pca(X_scaled, n_components=2)
print(f"PC1 Explained Variance: {var_ratio[0]*100:.2f}%")
print(f"PC2 Explained Variance: {var_ratio[1]*100:.2f}%")
print(f"Total Cumulative Variance: {cum_var[1]*100:.2f}%")


## 3. Elbow Method for Optimal K Selection

In [ ]:
elbow_df = compute_elbow(X_scaled, min_k=2, max_k=8)

plt.figure(figsize=(8, 4))
plt.plot(elbow_df["K"], elbow_df["Inertia"], marker="o", color="#00F2FE", linewidth=2)
plt.title("Elbow Method for K Selection")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("K-Means Inertia")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()


## 4. K-Means Model Training (K = 4)

In [ ]:
selected_k = 4
kmeans_model, cluster_labels = train_kmeans(X_scaled, n_clusters=selected_k, random_state=42)
df_clean["Cluster"] = cluster_labels
print("Cluster Label Counts:")
print(df_clean["Cluster"].value_counts())


## 5. Activity Behavior Group Profiling

In [ ]:
profile_df = profile_clusters(df_clean, "Cluster", summary["sensor_cols"], target_col=summary["target_col"])
display(profile_df[["Cluster", "Behavior Group", "Records", "Percentage (%)", "Dominant Activity", "Activity Level"]])


## 6. 2D PCA Cluster Scatter Plot

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    x=pca_df["PC1"],
    y=pca_df["PC2"],
    hue=[f"Cluster {c}" for c in cluster_labels],
    palette="viridis",
    alpha=0.7
)
plt.title("2D PCA Clusters Scatter Plot")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 7. Model Serialization & Artifact Saving

In [ ]:
save_ml_artifacts(
    models_dir="../models",
    kmeans_model=kmeans_model,
    scaler=scaler,
    pca_model=pca_model,
    feature_names=summary["sensor_cols"],
    cluster_profiles=profile_df
)
print("Models saved successfully into ../models/")
